# 05 — Modelo 2: Clasificación de Tipo de Anomalía

Entrenamos un segundo Random Forest que, dado que una fila ya fue detectada como anomalía por el Modelo 1, determina **qué tipo** de anomalía es.

El modelo distingue 6 tipos:
- Datos Faltantes, Sensor Atascado, Ruido, Valores Fuera de Rango, Desviación de Correlación, Contextual

**Entrada:** `data/interim/04_modelo1_predicciones.parquet` + modelos del paso anterior  
**Salida:** `data/models/modelo_2_clasificador.joblib`

In [ ]:
# ─── Intel Extension for Scikit-learn ────────────────────────────────────────
# IMPORTANTE: debe cargarse ANTES que cualquier import de sklearn.
# Instalar con: pip install scikit-learn-intelex
# En CPU Intel puede acelerar Random Forest e IterativeImputer 10-100x.
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    _intel_activo = True
except ImportError:
    _intel_activo = False

# ─── Librerías estándar ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, glob, joblib, pickle, warnings
import matplotlib.ticker as mticker

# ─── Scikit-learn (ya parcheado si Intel Extension está disponible) ───────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.experimental import enable_iterative_imputer  # requerido antes de IterativeImputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# ─── Configuración del proyecto ───────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import *

if _intel_activo:
    print("Intel Extension for Scikit-learn ACTIVA — algoritmos acelerados con oneAPI")
else:
    print("Tip: instala scikit-learn-intelex para acelerar en CPU Intel")
    print("     pip install scikit-learn-intelex")


## 0. Cargar datos y modelos del paso anterior

In [ ]:
# Cargar dataset con predicciones del Modelo 1
df_trabajo = pd.read_parquet(PARQUET_04)
print(f"Dataset cargado: {df_trabajo.shape}")

# Recuperar las mismas variables de entrenamiento del Modelo 1
excluir = COLUMNAS_EXCLUIR_FEATURES
columnas_features = [c for c in df_trabajo.columns if c not in excluir]
X = df_trabajo[columnas_features].select_dtypes(include='number').copy()
y = df_trabajo['etiqueta_deteccion'].copy()

# Reproducir el mismo split que en 04
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=TSCV_N_SPLITS)
splits = list(tscv.split(X))
train_idx, test_idx = splits[-1]   # último fold (mismo que en 04)
X_train = X.iloc[train_idx];  X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx];  y_test = y.iloc[test_idx]

# Cargar el imputer entrenado en 04 (no re-entrenar — evita leakage)
imputer = joblib.load(IMPUTER_M1_PATH)
feature_names = imputer.get_feature_names_out(X_train.columns)

# IMPORTANTE: preservar el índice original para que y_train/y_test y X_*_imputed estén alineados
X_train_imputed = pd.DataFrame(
    imputer.transform(X_train),
    columns=feature_names,
    index=X_train.index      # ← sin esto los índices no coinciden con y_train/y_test
)
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_names,
    index=X_test.index       # ← ídem
)
print("Imputación cargada y aplicada.")

## 1. Preparar datos de entrenamiento para Modelo 2
Filtramos solo las filas que **realmente son anomalías** en el conjunto de entrenamiento. El Modelo 2 solo ve anomalías — aprende a distinguir entre tipos, no entre normal y anomalía.

In [ ]:
if ('X_train_imputed' in locals() and 'y_train' in locals() and
    'X_test_imputed' in locals() and 'y_test' in locals() and
    'df_trabajo' in locals()):

    # --- Preparar datos de ENTRENAMIENTO para Modelo 2 ---
    # Filtrar solo las filas que son verdaderas anomalías en el conjunto de entrenamiento
    indices_anomalias_train = X_train_imputed[y_train == 'anomalia'].index
    X_model2_train = X_train_imputed.loc[indices_anomalias_train]

    # Obtener las etiquetas de tipo de anomalía correspondientes del df_trabajo original
    y_model2_train_str = df_trabajo.loc[indices_anomalias_train, 'etiqueta_tipo_anomalia']

    print(f"Forma de X_model2_train: {X_model2_train.shape}")
    if X_model2_train.empty:
        print("Advertencia: No se encontraron anomalías en el conjunto de entrenamiento para Modelo 2.")
    else:
        print("Distribución de tipos de anomalía en y_model2_train_str:")
        print(y_model2_train_str.value_counts())

    # --- Preparar datos de PRUEBA para Modelo 2 ---
    # Filtrar solo las filas que son verdaderas anomalías en el conjunto de prueba
    indices_anomalias_test = X_test_imputed[y_test == 'anomalia'].index
    X_model2_test = X_test_imputed.loc[indices_anomalias_test]

    # Obtener las etiquetas de tipo de anomalía correspondientes
    y_model2_test_str = df_trabajo.loc[indices_anomalias_test, 'etiqueta_tipo_anomalia']

    print(f"\nForma de X_model2_test: {X_model2_test.shape}")
    if X_model2_test.empty:
        print("Advertencia: No se encontraron anomalías en el conjunto de prueba para Modelo 2.")
    else:
        print("Distribución de tipos de anomalía en y_model2_test_str:")
        print(y_model2_test_str.value_counts())

    # --- Codificación de Etiquetas de Tipo de Anomalía ---
    if not X_model2_train.empty and not X_model2_test.empty:
        label_encoder_model2 = LabelEncoder()

        # Ajustar el codificador SOLO con las etiquetas de entrenamiento
        y_model2_train_encoded = label_encoder_model2.fit_transform(y_model2_train_str)

        # Transformar las etiquetas de prueba usando el codificador ya ajustado
        y_model2_test_encoded = label_encoder_model2.transform(y_model2_test_str)

        print("\nEtiquetas codificadas para Modelo 2:")
        print(f"Primeras 5 y_model2_train_encoded: {y_model2_train_encoded[:5]}")
        print(f"Primeras 5 y_model2_test_encoded: {y_model2_test_encoded[:5]}")

        clases_modelo2 = label_encoder_model2.classes_
        print(f"Clases del Modelo 2 (codificadas de 0 a {len(clases_modelo2)-1}): {clases_modelo2}")

        if pd.Series(y_model2_train_encoded).isnull().sum() > 0 or pd.Series(y_model2_test_encoded).isnull().sum() > 0:
            print("¡Advertencia! Se encontraron NaNs en las etiquetas codificadas del Modelo 2.")

    else:
        print("\nNo se procederá con la codificación de etiquetas ya que no hay suficientes datos de anomalías.")

else:
    print("Error: Datos necesarios para Modelo 2 no están definidos ('X_train_imputed', 'y_train', etc.).")

## 2. Entrenar Random Forest Clasificador de Tipos

In [ ]:
# Verificar que los datos para Modelo 2 existan y no estén vacíos
if ('X_model2_train' in locals() and not X_model2_train.empty and
    'y_model2_train_encoded' in locals() and len(y_model2_train_encoded) > 0):

    # Inicializar el clasificador Random Forest para Modelo 2
    # class_weight='balanced' es especialmente importante aquí, ya que el número de instancias de cada tipo de anomalía puede ser muy diferente.
    rf_clasificador_tipo = RandomForestClassifier(
        n_estimators=100,        # Número de árboles
        random_state=42,         # Para reproducibilidad
        class_weight='balanced', # Pondera las clases inversamente a su frecuencia
        n_jobs=-1                # Usar todos los procesadores
    )

    print("\nEntrenando el Modelo 2 (Clasificador de Tipos de Anomalía)...")
    rf_clasificador_tipo.fit(X_model2_train, y_model2_train_encoded)
    print("Entrenamiento del Modelo 2 completado.")

else:
    print("Error: Datos de entrenamiento para Modelo 2 (X_model2_train, y_model2_train_encoded) no están definidos o están vacíos.")
    # Detener la ejecución de los siguientes bloques si no hay datos para entrenar
    rf_clasificador_tipo = None

## 3. Evaluar Modelo 2

In [ ]:
if (rf_clasificador_tipo is not None and
    'X_model2_test' in locals() and not X_model2_test.empty and
    'y_model2_test_encoded' in locals() and len(y_model2_test_encoded) > 0 and
    'clases_modelo2' in locals()):

    print("\nRealizando predicciones con Modelo 2 en el conjunto de prueba...")
    y_pred_model2 = rf_clasificador_tipo.predict(X_model2_test)
    # y_pred_proba_model2 = rf_clasificador_tipo.predict_proba(X_model2_test) # Para métricas basadas en probabilidad si se necesitaran

    print("\n--- Evaluación del Modelo 2 (Clasificador de Tipos de Anomalía) ---")
    
    # Accuracy General
    accuracy_model2 = accuracy_score(y_model2_test_encoded, y_pred_model2)
    print(f"Accuracy General del Modelo 2: {accuracy_model2:.4f}")

    # Matriz de Confusión Multi-clase
    print("\nMatriz de Confusión del Modelo 2:")
    cm_model2 = confusion_matrix(y_model2_test_encoded, y_pred_model2)
    # print(cm_model2) # Imprimir la matriz numérica si se desea

    # Visualizar la matriz de confusión
    plt.figure(figsize=(10, 8)) # Ajustar tamaño según número de clases
    sns.heatmap(cm_model2, annot=True, fmt='d', cmap='Greens',
                xticklabels=clases_modelo2, yticklabels=clases_modelo2)
    plt.xlabel('Predicción del Tipo de Anomalía')
    plt.ylabel('Tipo de Anomalía Real')
    plt.title('Matriz de Confusión - Modelo 2 (Clasificador de Tipos)')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Reporte de Clasificación Detallado (Precisión, Recall, F1-score por clase)
    print("\nReporte de Clasificación del Modelo 2:")
    print(classification_report(y_model2_test_encoded, y_pred_model2, target_names=clases_modelo2))

else:
    print("Error: Modelo 2 no entrenado o datos de prueba para Modelo 2 no disponibles/vacíos.")
    if 'clases_modelo2' not in locals():
        print("  Además, 'clases_modelo2' (nombres de las clases) no está definido.")

## 4. Guardar Modelo 2

In [ ]:
# Guardar el clasificador de tipos y el LabelEncoder
os.makedirs(DATA_MODELS, exist_ok=True)
joblib.dump(rf_clasificador_tipo,  MODELO_2_PATH)
joblib.dump(label_encoder_model2,  LABEL_ENC_M2_PATH)
print(f"Modelo 2 guardado: {MODELO_2_PATH}")
print(f"Label Encoder guardado: {LABEL_ENC_M2_PATH}")
